In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import os


#  Device & Performance Setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


#  Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"   # change to your dataset
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")
print("Example class names:", dataset.classes[:10], "...")

# Stratified Split (85/10/5)
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

print("DataLoaders ready")


#  Model — ResNet-50

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)

num_classes = len(dataset.classes)  # 200 in your case
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device, memory_format=torch.channels_last)

if hasattr(torch, "compile"):
    model = torch.compile(model, mode="max-autotune")


#  Loss, Optimizer & Scheduler

criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4, nesterov=True)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


#  Training & Evaluation Loops

def evaluate_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type=="cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total
        val_acc = evaluate_model(model, val_loader, device)

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        scheduler.step()


#  Run Training

train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=20)


#  Save Final Weights

torch.save(model.state_dict(), "resnet50_weights.pth")
print("Saved ResNet-50 weights")


#  Final Test Accuracy

test_acc = evaluate_model(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2f}%")


Using device: cuda
Total images: 100000
Number of classes: 200
Example class names: ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750'] ...
DataLoaders ready


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/sapi279d/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s] 
Epoch 1/20:   0%|          | 0/665 [00:00<?, ?it/s]AUTOTUNE convolution(128x3x224x224, 64x3x7x7)
  convolution 2.1064 ms 100.0%
  triton_convolution_3 2.8600 ms 73.6%
  triton_convolution_4 3.1304 ms 67.3%
  triton_convolution_5 3.4253 ms 61.5%
  triton_convolution_2 4.1318 ms 51.0%
  triton_convolution_0 5.2931 ms 39.8%
  triton_convolution_1 9.3870 ms 22.4%
SingleProcess AUTOTUNE takes 51.1738 seconds
AUTOTUNE mm(401408x64, 64x64)
  triton_mm_14 0.0840 ms 100.0%
  triton_mm_8 0.0860 ms 97.6%
  triton_mm_7 0.0870 ms 96.5%
  triton_mm_6 0.0891 ms 94.3%
  triton_mm_10 0.0891 ms 94.3%
  triton_mm_13 0.0891 ms 94.3%
  triton_mm_9 0.0901 ms 93.2%
  triton_mm_15 0.0922 ms 91.1%
  mm 0.1004 ms 83.7%
  triton_mm_17 0.1157 ms 72.6%
SingleProcess AUTOTUNE takes 6.9418 seconds
AUTOTUNE convolution(128x64

Epoch [1/20] | Train Loss: 2.6064 | Train Acc: 51.37% | Val Acc: 46.14%


Epoch 2/20: 100%|██████████| 665/665 [02:03<00:00,  5.38it/s]


Epoch [2/20] | Train Loss: 2.0614 | Train Acc: 65.36% | Val Acc: 54.12%


...

Epoch [18/20] | Train Loss: 0.8893 | Train Acc: 99.92% | Val Acc: 72.71%


Epoch 19/20: 100%|██████████| 665/665 [02:00<00:00,  5.52it/s]


Epoch [19/20] | Train Loss: 0.8869 | Train Acc: 99.92% | Val Acc: 73.07%


Epoch 20/20: 100%|██████████| 665/665 [02:01<00:00,  5.47it/s]


Epoch [20/20] | Train Loss: 0.8857 | Train Acc: 99.94% | Val Acc: 72.96%
Saved ResNet-50 weights
Final Test Accuracy: 73.34%
